# Agnostic Conformal Audit -- Bring Your Own Quantile Path

This notebook demonstrates the **dependency-agnostic** audit path
introduced in `conformal-oracle` v0.2.0.  You supply a pre-computed
quantile forecast as a `pd.Series` and `conformal-oracle` handles
the conformal correction, regime classification, and backtesting --
no `arch`, no `torch`, no model library needed.

## Use cases

- You trained a VaR model in R, Julia, or a proprietary system and
  want to audit it in Python.
- You have production VaR forecasts stored in a database.
- You want to compare your model against baselines without installing
  heavy dependencies.

In [ ]:
!pip install -q conformal-oracle

In [ ]:
import numpy as np
import pandas as pd
from conformal_oracle import audit, classify_regime, compare_forecasters

np.random.seed(2026)

## 1. Generate synthetic data

We simulate GARCH(1,1) returns and create two competing quantile
forecasts: one well-calibrated and one deliberately mis-calibrated.

In [ ]:
rng = np.random.default_rng(2026)
n = 2000
omega, alpha_g, beta_g = 1e-6, 0.05, 0.90

returns = np.empty(n)
sigma2 = np.empty(n)
sigma2[0] = omega / (1 - alpha_g - beta_g)

for t in range(n):
    if t > 0:
        sigma2[t] = omega + alpha_g * returns[t-1]**2 + beta_g * sigma2[t-1]
    returns[t] = np.sqrt(sigma2[t]) * rng.standard_normal()

dates = pd.bdate_range('2018-01-02', periods=n)
r = pd.Series(returns, index=dates, name='returns')

# Well-calibrated 1% quantile (uses the true volatility)
from scipy import stats
q_lo_good = pd.Series(
    -np.sqrt(sigma2) * stats.norm.ppf(0.99),
    index=dates, name='q_lo_good',
)

# Mis-calibrated: assumes constant vol (too optimistic)
q_lo_bad = pd.Series(
    np.full(n, np.quantile(returns, 0.01)),
    index=dates, name='q_lo_bad',
)

print(f'{len(r)} observations')
print(f'Good forecast: q_lo mean = {q_lo_good.mean():.6f}')
print(f'Bad  forecast: q_lo mean = {q_lo_bad.mean():.6f}')

## 2. Static audit with `forecast=`

In [ ]:
result_good = audit(r, forecast=q_lo_good, alpha=0.01, mode='static')
print(result_good.summary())
print()
print(f'Note: z2 and fz are NaN because ES is not available from a quantile path.')

In [ ]:
result_bad = audit(r, forecast=q_lo_bad, alpha=0.01, mode='static')
print(result_bad.summary())

## 3. Regime classification

In [ ]:
verdict = classify_regime(r, forecast=q_lo_good, mode='static')
print(f'Regime: {verdict.regime}')
print(f'R = {verdict.R:.4f}')
print(f'Basel zone: {verdict.basel_zone}')

## 4. Compare two quantile paths

In [ ]:
comp = compare_forecasters(
    r,
    {'well_calibrated': q_lo_good, 'constant_vol': q_lo_bad},
    alpha=0.01,
    mode='static',
)
print(comp.comparison_table()[['violation_rate_corrected', 'quantile_score_corrected', 'regime']])
print()
print('DM matrix:')
print(comp.dm_matrix())

## 5. Rolling audit

In [ ]:
result_roll = audit(
    r, forecast=q_lo_good,
    alpha=0.01, mode='rolling',
    window=250, warmup=250,
)
print(result_roll.summary())